In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import logging
import h5py
import matplotlib.pyplot as plt  # type: ignore
import numpy as np
import healpy as hp
from itertools import product

from mlpng.utils import *
from mlpng.generator import Generator
from mlpng.utils.utils import *
from mlpng.utils.plots import plot_cl, plot_map, plot_map_alm

import lenspyx
from lenspyx import utils_hp

mpi_comm = None

In [ ]:
generator = Generator(
    [
        "settings/n128.json",
        "--nsims",
        "1",
        "--pols",
        "T",
        "--phi_scale",
        "2.0",
        "--shape",
        "local",
        "--no-noise",
        # "--double_precision",
    ]
)

# Get the unlensed gaussian Alms

This should be correct, notice we do not do any non-gaussian here due to just testing the lensing deflection field.

In [ ]:
pol_idxs = generator.pol_idxs()

sims = [
    hp.synalm(generator.c_ell, lmax=generator.lmax, new=True)
    for _ in range(generator.nsims)
]
sims = remove_mono_dipole(np.array(sims))
alm_l = np.ascontiguousarray(sims)[0, 0]  # ensure contiguous memory

plot_cl_alm(
    generator,
    alm_l,
    title=r"Unlensed gaussian $C_{\ell}^{TT}$",
    ylabel=r"$C_{\ell}^{TT}$",
    show=True,
    plot_camb=True,
)
plot_map_alm(
    generator,
    alm_l,
    title=r"Unlensed gaussian $C_{\ell}^{TT}$",
    zoom=True,
    show=True,
)

# Deflection Fields

Here we generate the deflection fields $d_{\ell m}$ for the lensing. We get the lensing potential power spectrum $C_{\ell}^{\phi\phi}$ from CAMB, and convert this to the deflection field by $d_{\ell m} = \sqrt{\ell(\ell+1)}C_{\ell}^{\phi\phi}$. This is per the lenspyx demo https://github.com/carronj/lenspyx/blob/master/examples/demo_lenspyx.ipynb , but I would appricate confirmation that the $C_{\ell}^{\phi\phi}$ is the correct one to use here.

To scale the lensing potential I am currently applying the scale to the $p_{lm}$ before converting to the harmoincs. This is disable right now as I am testing just a normal lensing potential, but does that look correct?

In [ ]:
# we use a buffer for the lmax to ensure we can compute the lensing potential without error
lmax = generator.lmax + generator.lmax_buffer

# our conversion factor = sqrt(l * (l + 1))
fl = np.sqrt(np.arange(lmax + 1) * np.arange(1, lmax + 2))

# cl_phi is PP PT PE, we only want the phi phi part
cl_phi = generator.cosmo._camb_data.get_lens_potential_cls(
    lmax,
    "muK",
    raw_cl=True,
)
cl_phi = cl_phi[..., 0]

# the weird factors here are to match plots from lewis
# https://indico.cern.ch/event/218615/contributions/448794/attachments/353290/492159/Lewis.pdf page 11
plot_cl(
    generator,
    cl_phi * fl**4 / 2 / np.pi,
    title=r"Lensing potential $C_{\ell}^{\phi\phi}$",
    ylabel=r"$(\ell (\ell + 1))^2 / 2 \pi\;C_{\ell}^{\phi\phi}$",
    show=True,
    scale=False,
    plot_func=plt.loglog,
)


In [ ]:
alm_phi = hp.synalm(
    cl_phi,
    lmax,
    lmax,
    new=True,
    verbose=False,
)
alm_phi *= generator.phi_scale  # scale the lensing potential by phi_scale

plot_cl_alm(
    generator,
    alm_phi,
    title="Lensing potential alm, $\phi_{lm}$",
    ylabel=r"$\phi_{lm}$",
    show=True,
    # scale=False,
    plot_func=plt.loglog,
)
plot_map_alm(generator, alm_phi, title="plm", zoom=True, show=True)


In [ ]:
# transform the lensing potential into spin-1 deflection field
dlm = hp.almxfl(alm_phi, fl)

plot_cl_alm(
    generator,
    dlm,
    lmax=lmax,
    title="Lensing deflection field alm",
    ylabel=r"$d_{lm}$",
    # scale=False
    show=True,
    plot_func=plt.loglog,
)
plot_map_alm(generator, dlm, title="$d_{lm}$", zoom=True, show=True)

here we get the lensed maps from the gaussian alm's using our $d_{lm}$s calculated previously. We only do a single T mode lenmap no matter the settings above for testing, this gives us the `alm_l[0, 0]` below which is dimensions `(sim, pol, data)`.

In [ ]:
lenmap = lenspyx.alm2lenmap(
    alm_l,
    dlm,
    geometry=("healpix", {"nside": generator.nside}),
    nthreads=generator.slurm.n_cpus,
)
lenmap = remove_mono_dipole(lenmap)

plot_map(lenmap, title="Lensed map", zoom=True, show=True)

In [ ]:
unlen_map = hp.alm2map(
    alm_l,
    nside=generator.nside,
    lmax=generator.lmax,
    verbose=False,
)
unlen_map = remove_mono_dipole(unlen_map)

plot_map(unlen_map, title="Unlensed map", zoom=True, show=True)
plot_map(lenmap - unlen_map, title="Lensed - Unlensed map", zoom=True, show=True)

Here is where we caculate the error in our estimator for the lenses code. To get the $C_{\ell}$ used to get the $\text{icov}=C^{-1}$ used in the calculation I am using the lensed maps and using anafast to calculate the cls. With many maps I would average over the resulting cls. The issues here is a higher lensing potential will result in a lower variance which is not what we expect.

In [ ]:
cl_lens = hp.anafast(lenmap, use_pixel_weights=True)
cl_lens[:2] = 0

plot_cl(generator, cl_lens, title="Lensed C_l", show=True, plot_camb=True)

plot_cl_vs(
    generator,
    generator.c_ell[0],
    cl_lens,
    title="Lensed vs Unlensed C_l",
    ylabel=r"$C_{\ell}^{TT}$",
    show=True,
    # scale=False,
    # plot_func=plt.plot,
)

plot_cl(
    generator,
    cl_lens / generator.c_ell,
    title="Lensed C_l / Unlensed C_l",
    ylabel=r"$\frac{C_{\ell}^{TT}}{C_{\ell}^{TT,unlensed}}$",
    show=True,
    scale=False,
    plot_func=plt.plot,
)

plot_cl(
    generator,
    (cl_lens - generator.c_ell) / generator.c_ell,
    title="(Lensed - Unlensed C_l) / Unlensed C_l",
    ylabel=r"$\frac{C_{\ell}^{TT} - C_{\ell}^{TT,unlensed}}{C_{\ell}^{TT,unlensed}}$",
    show=True,
    scale=False,
    plot_func=plt.plot,
)

# Compute the fishers

Here we compute the fishers, this takes a very long time based on the nside used

In [ ]:
fisher_mat = np.array(generator.compute_fisher_shapes(generator.shapes))
if len(generator.shapes) > 1:
    marg_likes = np.sqrt(np.diag(np.linalg.inv(fisher_mat)))
else:
    marg_likes = np.sqrt(1 / fisher_mat)

print(f"Unlensed marginal likelihoods: {marg_likes}")

In [ ]:
lens_cls = generator.cosmo.c_ell["lensed_scalar"]["c_ell"][: generator.nell, 0]
lens_cls *= generator.phi_scale

plot_cl_vs(generator, lens_cls, cl_lens, labels=["camb", "lens"], show=True)

In [ ]:
icov = 1 / lens_cls
icov[: generator.lmin] = 0.0
icov = np.array(icov)

fisher_mat = generator.compute_fisher_shapes(generator.shapes, icov=icov)
if len(generator.shapes) > 1:
    marg_likes_lens = np.sqrt(np.diag(np.linalg.inv(fisher_mat)))
else:
    marg_likes_lens = np.sqrt(1 / fisher_mat)

print(f"Marginal likelihoods: {marg_likes_lens}")
print(f"diff: {marg_likes_lens - marg_likes}")

In [ ]:
diff = marg_likes_lens - marg_likes
ratio = marg_likes_lens / marg_likes

print("nside:", generator.nside)
print(
    f"| {generator.phi_scale} | {float(marg_likes)} | {float(marg_likes_lens)} | {float(diff)} | {float(ratio)} |"
)

nside: 128
| phi | unlensed | lensed | diff | ratio |
| --- | --- | --- | --- | --- |
| 1.0 | 22.077037811279297 | 22.080591201782227 | 0.0035533905029296875 | 1.0001609325408936 |
| 2.0 | 22.07703971862793 | 62.45334243774414 | 40.376304626464844 | 2.8288819789886475 |


# the big loop

Just do a big loop to get a table, will take a long time

In [ ]:
# disable a root logger error from CAMB
logging.getLogger("root").setLevel(logging.ERROR)

for phi in [1.0, 1.1, 2, 3, 5, 10, 30, 100]:
    generator = Generator(
        [
            "settings/n64.json",
            "--nsims",
            "1",
            "--pols",
            "T",
            "--phi_scale",
            str(phi),
            "--shape",
            "local",
            "--no-noise",
            # "--double_precision",
        ],
        logging.ERROR,
    )

    pol_idxs = generator.pol_idxs()

    sims = [
        hp.synalm(generator.c_ell, lmax=generator.lmax, new=True)
        for _ in range(generator.nsims)
    ]
    sims = remove_mono_dipole(np.array(sims))
    alm_l = np.ascontiguousarray(sims)[0, 0]  # ensure contiguous memory

    # we use a buffer for the lmax to ensure we can compute the lensing potential without error
    lmax = generator.lmax + generator.lmax_buffer

    # our conversion factor = sqrt(l * (l + 1))
    fl = np.sqrt(np.arange(lmax + 1) * np.arange(1, lmax + 2))

    # cl_phi is PP PT PE, we only want the phi phi part
    cl_phi = generator.cosmo._camb_data.get_lens_potential_cls(
        lmax,
        "muK",
        raw_cl=True,
    )
    cl_phi = cl_phi[..., 0]

    alm_phi = hp.synalm(
        cl_phi,
        lmax,
        lmax,
        new=True,
        verbose=False,
    )
    alm_phi *= generator.phi_scale  # scale the lensing potential by phi_scale

    dlm = hp.almxfl(alm_phi, fl)

    lenmap = lenspyx.alm2lenmap(
        alm_l,
        dlm,
        geometry=("healpix", {"nside": generator.nside}),
        nthreads=generator.slurm.n_cpus,
    )
    lenmap = remove_mono_dipole(lenmap)

    lens_cls = generator.cosmo.c_ell["lensed_scalar"]["c_ell"][: generator.nell, 0]
    lens_cls *= generator.phi_scale

    icov = 1 / lens_cls
    icov[: generator.lmin] = 0.0
    icov = np.array(icov)

    fisher_mat = np.array(generator.compute_fisher_shapes(generator.shapes))
    if len(generator.shapes) > 1:
        marg_likes = np.sqrt(np.diag(np.linalg.inv(fisher_mat)))
    else:
        marg_likes = np.sqrt(1 / fisher_mat)

    fisher_mat = generator.compute_fisher_shapes(generator.shapes, icov=icov)
    if len(generator.shapes) > 1:
        marg_likes_lens = np.sqrt(np.diag(np.linalg.inv(fisher_mat)))
    else:
        marg_likes_lens = np.sqrt(1 / fisher_mat)

    diff = marg_likes_lens - marg_likes
    ratio = marg_likes_lens / marg_likes

    print(
        f"| {generator.phi_scale} | {float(marg_likes)} | {float(marg_likes_lens)} | {float(diff)} | {float(ratio)} |"
    )


nside: 32
| phi | unlensed | lensed | diff | ratio |
| --- | --- | --- | --- | --- |
| 1.0 | 85.31002807617188 | 85.39151000976562 | 0.08148193359375 | 1.0009551048278809 |
| 1.1 | 85.3100357055664 | 98.51531982421875 | 13.205284118652344 | 1.1547917127609253 |
| 2.0 | 85.31002807617188 | 241.52366638183594 | 156.21363830566406 | 2.8311285972595215 |
| 3.0 | 85.3100357055664 | 443.707275390625 | 358.3972473144531 | 5.201114654541016 |
| 5.0 | 85.3100357055664 | 954.7061767578125 | 869.3961181640625 | 11.191018104553223 |
| 10.0 | 85.3100357055664 | 2700.31689453125 | 2615.0068359375 | 31.65298080444336 |
| 100.0 | 85.3100357055664 | 85391.5234375 | 85306.2109375 | 1000.9552001953125 |


# big run 

Nside: 32
| phi | unlensed | lensed | diff | ratio |
| --- | --- | --- | --- | --- |
| 1.0 | 85.3100357055664 | 85.39151000976562 | 0.08147430419921875 | 1.0009549856185913 |
| 1.1 | 85.3100357055664 | 98.51531982421875 | 13.205284118652344 | 1.1547917127609253 |
| 2.0 | 85.3100357055664 | 241.523681640625 | 156.21365356445312 | 2.8311285972595215 |
| 3.0 | 85.3100357055664 | 443.70733642578125 | 358.3973083496094 | 5.201115131378174 |
| 5.0 | 85.31002807617188 | 954.7061767578125 | 869.3961181640625 | 11.191019058227539 |
| 10.0 | 85.31002807617188 | 2700.31689453125 | 2615.0068359375 | 31.652982711791992 |
| 30.0 | 85.3100357055664 | 14031.255859375 | 13945.9462890625 | 164.4736785888672 |
| 100.0 | 85.3100357055664 | 85391.53125 | 85306.21875 | 1000.9552612304688 |

Nside: 64
| phi | unlensed | lensed | diff | ratio |
| --- | --- | --- | --- | --- |
| 1.0 | 43.28276443481445 | 43.302616119384766 | 0.0198516845703125 | 1.0004585981369019 |
| 1.1 | 43.28276443481445 | 49.957786560058594 | 6.675022125244141 | 1.1542189121246338 |
| 2.0 | 43.28276443481445 | 122.4782943725586 | 79.19552612304688 | 2.8297243118286133 |
| 3.0 | 43.28276443481445 | 225.00698852539062 | 181.72422790527344 | 5.198535442352295 |
| 5.0 | 43.28276443481445 | 484.1379699707031 | 440.8551940917969 | 11.185467720031738 |
| 10.0 | 43.28276443481445 | 1369.348876953125 | 1326.066162109375 | 31.637279510498047 |
| 30.0 | 43.28276443481445 | 7115.34521484375 | 7072.0625 | 164.39212036132812 |
| 100.0 | 43.28276443481445 | 43302.60546875 | 43259.32421875 | 1000.4584350585938 |

Nside: 128
| phi | unlensed | lensed | diff | ratio |
| --- | --- | --- | --- | --- |
| 1.0 | 22.07703971862793 | 22.080591201782227 | 0.003551483154296875 | 1.000160813331604 |
| 1.1 | 22.07703971862793 | 25.47415542602539 | 3.397115707397461 | 1.153875470161438 |
| 2.0 | 22.07703971862793 | 62.453338623046875 | 40.37629699707031 | 2.8288819789886475 |
| 3.0 | 22.07703971862793 | 114.7341079711914 | 92.65706634521484 | 5.196988105773926 |
| 5.0 | 22.07703971862793 | 246.86846923828125 | 224.7914276123047 | 11.182136535644531 |
| 10.0 | 22.07703971862793 | 698.2494506835938 | 676.1724243164062 | 31.627857208251953 |
| 30.0 | 22.077037811279297 | 3628.210205078125 | 3606.133056640625 | 164.34315490722656 |
| 100.0 | 22.07703971862793 | 22080.5859375 | 22058.509765625 | 1000.16064453125 |

Nside: 256
| phi | unlensed | lensed | diff | ratio |
| --- | --- | --- | --- | --- |
| 1.0 | 9.953892707824707 | 9.973581314086914 | 0.01968860626220703 | 1.0019780397415161 |
| 1.1 | 9.953892707824707 | 11.506417274475098 | 1.5525245666503906 | 1.155971646308899 |
| 2.0 | 9.953892707824707 | 28.209550857543945 | 18.255657196044922 | 2.834022045135498 |
| 3.0 | 9.953892707824707 | 51.82440948486328 | 41.87051773071289 | 5.206446647644043 |
| 5.0 | 9.953892707824707 | 111.5079345703125 | 101.55403900146484 | 11.202445030212402 |
| 10.0 | 9.953892707824707 | 315.39208984375 | 305.4382019042969 | 31.685300827026367 |
| 30.0 | 9.953892707824707 | 1638.826904296875 | 1628.873046875 | 164.64181518554688 |
| 100.0 | 9.953892707824707 | 9973.6083984375 | 9963.654296875 | 1001.980712890625 |
